# 01 — Data Collection

Runs the Reddit scraper and stock downloader, then does basic sanity checks on what came back.

**Run this before any other notebook.**

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from utils import EVENT_DATES, TICKERS

## 1a. Reddit collection

This calls `reddit_scraper.run_all()`. It will skip events already collected.
Expect ~5–10 minutes per event on first run.

In [ ]:
import reddit_scraper
reddit_scraper.run_all()

## 1b. Check Reddit data

In [ ]:
reddit_dir = Path('../data/raw/reddit')

for event in EVENT_DATES:
    path = reddit_dir / f'{event}.csv'
    if path.exists():
        df = pd.read_csv(path, parse_dates=['date'])
        print(f'{event}: {len(df)} posts  |  date range: {df.date.min()} → {df.date.max()}')
        print(f'  subreddits: {df.subreddit.value_counts().to_dict()}')
        print()
    else:
        print(f'{event}: NO FILE')

## 1c. Post volume over time (per event)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, event in enumerate(EVENT_DATES):
    path = reddit_dir / f'{event}.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path, parse_dates=['date'])
    daily = df.groupby('date').size()
    event_date = EVENT_DATES[event]

    axes[i].bar(daily.index, daily.values, color='#B5933A', alpha=0.8)
    axes[i].axvline(pd.Timestamp(event_date), color='red', linestyle='--', label='Event')
    axes[i].set_title(event.replace('_', ' ').title())
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Posts')
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Reddit Post Volume Around Each Event', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/post_volume.png', dpi=150, bbox_inches='tight')
plt.show()

## 1d. Stock data download

In [ ]:
import stock_data
stock_data.run()

## 1e. Check stock data

In [ ]:
prices = pd.read_csv('../data/raw/stocks/prices.csv', index_col=0, parse_dates=True)
print(f'Prices shape: {prices.shape}')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'\nMissing values:\n{prices.isnull().sum()}')
prices.tail()